# data-cleaning_008

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (219).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `crypto` |
| Trading style | `unknown` |
| Timeframe | `5m` |
| Code cells | 5 |
| Detected functions | validate, __init__, _get_symbol_info, _retry, get_account, get_balance, get_open_orders, cancel_all_orders, round_price, round_quantity, submit_market_order, confirm_order_filled |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
!pip install loguru
!pip install binance
!pip install python-binance

In [ ]:
import pandas as pd
import numpy as np
import time
import json
import os
from datetime import datetime, timedelta, timezone
from binance.client import Client
from binance.enums import *
from loguru import logger
from google.colab import drive
from google.colab import userdata
import math
from typing import Any, Callable, Optional, Union, List, Tuple
import requests

# Mount Google Drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/My Drive/crypto_trader/'
os.makedirs(DRIVE_PATH, exist_ok=True)

# ===================== Configuration =====================
class TradingConfig:
    try:
        # For testnet, use different credential names to avoid confusion
        if True:  # TESTNET flag
            API_KEY = userdata.get('BINANCE_TESTNET_API_KEY') or userdata.get('BINANCE_API_KEY')
            API_SECRET = userdata.get('BINANCE_TESTNET_SECRET_KEY') or userdata.get('BINANCE_SECRET_KEY')
        else:
            API_KEY = userdata.get('BINANCE_API_KEY')
            API_SECRET = userdata.get('BINANCE_SECRET_KEY')
    except:
        if True:  # TESTNET flag
            API_KEY = os.environ.get('BINANCE_TESTNET_API_KEY', input("Enter BINANCE_TESTNET_API_KEY: "))
            API_SECRET = os.environ.get('BINANCE_TESTNET_SECRET_KEY', input("Enter BINANCE_TESTNET_SECRET_KEY: "))
        else:
            API_KEY = os.environ.get('BINANCE_API_KEY', input("Enter BINANCE_API_KEY: "))
            API_SECRET = os.environ.get('BINANCE_SECRET_KEY', input("Enter BINANCE_SECRET_KEY: "))

    SYMBOL = "XRPUSDT"  # Binance uses USDT pairs
    BASE_URL = "https://testnet.binance.vision"  # Testnet URL
    TESTNET = True  # Set to False for live trading
    CAPITAL = 10000
    TAKE_PROFIT_PCT = 0.015
    STOP_LOSS_PCT = 0.005
    ENTRY_STOP_LOSS_PCT = 0.03
    BARS_LOOKBACK = 3000
    POSITION_STATE_FILE = os.path.join(DRIVE_PATH, "position_state_5min_binance.json")
    LOG_FILE = os.path.join(DRIVE_PATH, "crypto_trader_5min_binance.log")
    GRACE_SECONDS = 2
    ORDER_CONFIRM_TIMEOUT = 5
    ORDER_RETRY_DELAY = 2
    MIN_PRICE = 0.01
    MIN_STD = 1e-6
    PRICE_DECIMALS = 4
    QTY_DECIMALS = 2
    FEE_RATE = 0.001  # Binance spot trading fee
    MIN_QTY = 1.0  # Minimum quantity for XRP
    MIN_NOTIONAL = 10.0  # Minimum order value in USDT

    @staticmethod
    def validate():
        if not (TradingConfig.API_KEY and TradingConfig.API_SECRET):
            raise ValueError("API credentials not set.")

# ===================== Logger Setup =====================
logger.remove()
logger.add(
    TradingConfig.LOG_FILE,
    rotation="10 MB",
    level="DEBUG",
    format="{time:YYYY-MM-DD HH:mm:ss} [{level}] {message}"
)
logger.add(
    sink=lambda msg: print(msg, end=""),
    level="INFO",
    format="{time:YYYY-MM-DD HH:mm:ss} [{level}] {message}"
)

# ===================== Custom Exceptions =====================
class InsufficientBalanceError(Exception):
    pass

class MinNotionalError(Exception):
    pass

# ===================== Binance API Client =====================
class BinanceClient:
    def __init__(self, client=None, retry_defaults=None):
        TradingConfig.validate()
        self.config = TradingConfig

        if client:
            self.client = client
        else:
            if self.config.TESTNET:
                self.client = Client(
                    self.config.API_KEY,
                    self.config.API_SECRET,
                    testnet=True
                )
            else:
                self.client = Client(
                    self.config.API_KEY,
                    self.config.API_SECRET
                )

        self.retry_defaults = retry_defaults or dict(max_attempts=3, backoff_base=0.3, backoff_factor=2.0)

        # Get symbol info for precision
        self._get_symbol_info()

    def _get_symbol_info(self):
        """Get symbol information for precision settings"""
        try:
            exchange_info = self.client.get_exchange_info()
            for symbol in exchange_info['symbols']:
                if symbol['symbol'] == self.config.SYMBOL:
                    # Get price and quantity precision
                    for filter_info in symbol['filters']:
                        if filter_info['filterType'] == 'PRICE_FILTER':
                            tick_size = float(filter_info['tickSize'])
                            self.price_precision = len(str(tick_size).rstrip('0').split('.')[-1])
                        elif filter_info['filterType'] == 'LOT_SIZE':
                            step_size = float(filter_info['stepSize'])
                            self.qty_precision = len(str(step_size).rstrip('0').split('.')[-1])
                        elif filter_info['filterType'] == 'MIN_NOTIONAL':
                            self.config.MIN_NOTIONAL = float(filter_info['minNotional'])
                    break

            logger.info(f"Symbol info - Price precision: {self.price_precision}, Qty precision: {self.qty_precision}")

        except Exception as e:
            logger.warning(f"Could not get symbol info: {e}. Using defaults.")
            self.price_precision = self.config.PRICE_DECIMALS
            self.qty_precision = self.config.QTY_DECIMALS

    def _retry(self, fn, *args, **kwargs):
        params = {
            "max_attempts": kwargs.pop("max_attempts", self.retry_defaults["max_attempts"]),
            "backoff_base": kwargs.pop("backoff_base", self.retry_defaults["backoff_base"]),
            "backoff_factor": kwargs.pop("backoff_factor", self.retry_defaults["backoff_factor"]),
        }
        attempt, last_exc = 0, None
        while attempt < params["max_attempts"]:
            attempt += 1
            try:
                return fn(*args, **kwargs)
            except Exception as exc:
                last_exc = exc
                wait = params["backoff_base"] * (params["backoff_factor"] ** (attempt - 1))
                logger.debug(
                    f"[{self.config.SYMBOL}] attempt {attempt}/{params['max_attempts']} failed: "
                    f"{type(exc).__name__} - {exc}; retrying in {wait:.2f}s",
                    exc_info=True
                )
                time.sleep(wait)
        logger.error(f"[{self.config.SYMBOL}] All {params['max_attempts']} attempts failed for {fn.__name__}")
        raise last_exc

    def get_account(self):
        try:
            account = self.client.get_account()
            # Verify we can access account data
            if 'balances' not in account:
                raise ValueError("Invalid account response")
            return account
        except Exception as e:
            error_msg = str(e)
            if "Invalid API-key" in error_msg or "code=-2015" in error_msg:
                logger.error(f"[{self.config.SYMBOL}] API Key Error: {error_msg}")
                logger.error(f"[{self.config.SYMBOL}] Please check:")
                logger.error(f"  1. API key permissions (Enable Reading + Spot Trading)")
                logger.error(f"  2. For testnet: Use testnet.binance.vision credentials")
                logger.error(f"  3. Remove IP restrictions or whitelist your IP")
            else:
                logger.error(f"[{self.config.SYMBOL}] Failed to fetch account details: {type(e).__name__}: {e}", exc_info=True)
            return None

    def get_balance(self, asset):
        """Get balance for a specific asset"""
        try:
            account = self.get_account()
            if account:
                for balance in account['balances']:
                    if balance['asset'] == asset:
                        return float(balance['free']), float(balance['locked'])
            return 0.0, 0.0
        except Exception as e:
            logger.error(f"[{self.config.SYMBOL}] Failed to get balance for {asset}: {e}", exc_info=True)
            return 0.0, 0.0

    def get_open_orders(self, symbol: str):
        try:
            orders = self.client.get_open_orders(symbol=symbol)
            return orders or []
        except Exception as e:
            logger.error(f"[{symbol}] Failed to fetch open orders: {type(e).__name__}: {e}", exc_info=True)
            return []

    def cancel_all_orders(self, symbol: str):
        try:
            orders = self.get_open_orders(symbol)
            for order in orders:
                try:
                    self.client.cancel_order(symbol=symbol, orderId=order['orderId'])
                    logger.info(f"[{symbol}] Canceled order {order['orderId']}")
                except Exception as inner:
                    logger.warning(f"[{symbol}] Failed to cancel order {order['orderId']}: {inner}", exc_info=True)
            return True
        except Exception as e:
            logger.error(f"[{symbol}] Failed to cancel orders: {type(e).__name__}: {e}", exc_info=True)
            return False

    def round_price(self, price):
        """Round price to appropriate precision"""
        return round(price, self.price_precision)

    def round_quantity(self, quantity):
        """Round quantity to appropriate precision"""
        return round(quantity, self.qty_precision)

    def submit_market_order(self, symbol: str, qty: float, side: str):
        try:
            # Round quantity and validate
            qty = self.round_quantity(qty)

            if qty < self.config.MIN_QTY:
                raise ValueError(f"Quantity {qty} below minimum {self.config.MIN_QTY}")

            # Get current price to check notional value
            ticker = self.client.get_symbol_ticker(symbol=symbol)
            current_price = float(ticker['price'])
            notional_value = qty * current_price

            if notional_value < self.config.MIN_NOTIONAL:
                raise MinNotionalError(f"Notional value {notional_value} below minimum {self.config.MIN_NOTIONAL}")

            side_enum = SIDE_BUY if side.lower() == 'buy' else SIDE_SELL

            order = self._retry(
                self.client.order_market,
                symbol=symbol,
                side=side_enum,
                quantity=qty
            )

            logger.info(f"[{symbol}] Market {side} order placed: qty={qty}, id={order.get('orderId')}")
            return order

        except Exception as e:
            error_msg = str(e)
            if "insufficient balance" in error_msg.lower():
                raise InsufficientBalanceError(error_msg)
            elif "min notional" in error_msg.lower():
                raise MinNotionalError(error_msg)
            logger.error(f"[{symbol}] Market order failed: {type(e).__name__}: {error_msg}")
            raise

    def confirm_order_filled(self, symbol: str, order_id: str, timeout=None):
        timeout = timeout or self.config.ORDER_CONFIRM_TIMEOUT
        start_time = time.time()

        while time.time() - start_time < timeout:
            try:
                order = self.client.get_order(symbol=symbol, orderId=order_id)
                status = order.get('status', '').lower()

                if status == 'filled':
                    logger.info(f"[{symbol}] Order {order_id} filled")
                    return True
                elif status in ['canceled', 'cancelled', 'rejected', 'expired']:
                    logger.warning(f"[{symbol}] Order {order_id} {status}")
                    return False

            except Exception as e:
                logger.error(f"[{symbol}] Failed to check order {order_id} status: {e}", exc_info=True)

            time.sleep(0.5)

        logger.warning(f"[{symbol}] Order {order_id} confirmation timed out after {timeout}s")
        return False

    def get_latest_price(self, symbol: str):
        try:
            ticker = self.client.get_symbol_ticker(symbol=symbol)
            return float(ticker['price'])
        except Exception as e:
            logger.warning(f"[{symbol}] Price fetch failed: {e}", exc_info=True)
            return 0.0

    def get_orderbook(self, symbol: str):
        """Get orderbook for bid/ask prices"""
        try:
            depth = self.client.get_order_book(symbol=symbol, limit=5)
            bid = float(depth['bids'][0][0]) if depth['bids'] else 0.0
            ask = float(depth['asks'][0][0]) if depth['asks'] else 0.0
            return bid, ask
        except Exception as e:
            logger.warning(f"[{symbol}] Orderbook fetch failed: {e}", exc_info=True)
            return 0.0, 0.0

    def test_connection(self):
        """Test API connection and permissions"""
        try:
            logger.info(f"Testing connection to {'testnet' if self.config.TESTNET else 'mainnet'}...")

            # Test server time
            server_time = self.client.get_server_time()
            logger.info(f"Server time: {server_time}")

            # Test account access
            account = self.get_account()
            if account:
                logger.info("✅ Account access successful")

                # Show balances
                for balance in account['balances']:
                    free = float(balance['free'])
                    locked = float(balance['locked'])
                    if free > 0 or locked > 0:
                        logger.info(f"  {balance['asset']}: Free={free}, Locked={locked}")
                return True
            else:
                logger.error("❌ Account access failed")
                return False

        except Exception as e:
            logger.error(f"❌ Connection test failed: {e}")
            return False

# ===================== State Management =====================
class PositionState:
    @staticmethod
    def load():
        if os.path.exists(TradingConfig.POSITION_STATE_FILE):
            try:
                with open(TradingConfig.POSITION_STATE_FILE, "r") as f:
                    state = json.load(f)
                    if not isinstance(state, dict):
                        raise ValueError("Invalid state format")
                    return state
            except Exception as e:
                logger.warning(f"Position state corrupted: type={type(e).__name__}, message={str(e)}. Starting fresh.", exc_info=True)
        return {}

    @staticmethod
    def save(state):
        tmp_file = TradingConfig.POSITION_STATE_FILE + ".tmp"
        try:
            with open(tmp_file, "w") as f:
                json.dump(state, f, indent=2)
            os.replace(tmp_file, TradingConfig.POSITION_STATE_FILE)
        except Exception as e:
            logger.error(f"Failed to save position state: type={type(e).__name__}, message={str(e)}", exc_info=True)
            raise

    @staticmethod
    def clear(symbol):
        state = PositionState.load()
        if symbol in state:
            del state[symbol]
            try:
                PositionState.save(state)
                logger.info(f"[{symbol}] Position state cleared.")
            except Exception as e:
                logger.error(f"[{symbol}] Failed to clear position state: {type(e).__name__}: {str(e)}", exc_info=True)

    @staticmethod
    def update(symbol, data):
        state = PositionState.load()
        state[symbol] = data
        try:
            PositionState.save(state)
        except Exception as e:
            logger.error(f"[{symbol}] Failed to update position state: {type(e).__name__}: {str(e)}", exc_info=True)

# ===================== Data Fetching =====================
def fetch_bars(client, symbol, bars=TradingConfig.BARS_LOOKBACK):
    """Fetch kline data from Binance"""
    MAX_ATTEMPTS = 3
    MIN_BARS = 60
    TARGET_BARS = bars
    SLEEP_BETWEEN_CALLS = 1
    SLEEP_ON_FAIL = 30

    for attempt in range(MAX_ATTEMPTS):
        try:
            logger.info(f"[{symbol}] Fetching {TARGET_BARS} 5-minute bars (attempt {attempt+1})")

            # Get klines from Binance (5m interval)
            klines = client.client.get_historical_klines(
                symbol=symbol,
                interval=Client.KLINE_INTERVAL_5MINUTE,
                limit=TARGET_BARS
            )

            if not klines or len(klines) < MIN_BARS:
                logger.warning(f"[{symbol}] Insufficient klines: {len(klines) if klines else 0}")
                if attempt < MAX_ATTEMPTS - 1:
                    time.sleep(SLEEP_ON_FAIL)
                    continue
                return None

            # Convert to DataFrame
            df = pd.DataFrame(klines, columns=[
                'timestamp', 'open', 'high', 'low', 'close', 'volume',
                'close_time', 'quote_volume', 'count', 'taker_buy_volume',
                'taker_buy_quote_volume', 'ignore'
            ])

            # Select and convert required columns
            df = df[['timestamp', 'open', 'high', 'low', 'close']].copy()
            df['time'] = pd.to_datetime(df['timestamp'], unit='ms').dt.tz_localize('UTC')

            # Convert price columns to float
            for col in ['open', 'high', 'low', 'close']:
                df[col] = df[col].astype(float)

            df = df.drop('timestamp', axis=1)
            df = df[['time', 'open', 'high', 'low', 'close']]
            df.sort_values('time', inplace=True)
            df.reset_index(drop=True, inplace=True)

            logger.info(f"[{symbol}] Successfully fetched {len(df)} bars")
            return df

        except Exception as e:
            logger.error(f"[{symbol}] Attempt {attempt+1}/{MAX_ATTEMPTS} failed: {e}")
            if attempt < MAX_ATTEMPTS - 1:
                time.sleep(SLEEP_ON_FAIL)
            else:
                return None

# ===================== Indicator Calculation =====================
def compute_jma_signal(
    df: pd.DataFrame,
    smooth_length: int = 14,
    norm_period: int = 60,
    upper_threshold: float = 0.5,
    lower_threshold: float = -0.5,
    rs_period: int = 14,
    rs_percentile: float = 10.0,
    min_price: float = 1e-12,
    min_std: float = 1e-8,
    symbol: str = "SYMBOL",
    logger=None
) -> pd.DataFrame:
    log = logger.info if logger else print
    warn = logger.warning if logger else print
    error = logger.error if logger else print
    debug = logger.debug if logger else print

    try:
        required_cols = ["open", "high", "low", "close", "time"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            error(f"[{symbol}] Missing required columns: {missing}")
            return None

        if len(df) < norm_period:
            error(f"[{symbol}] Insufficient data: {len(df)} bars, need at least {norm_period}")
            return None

        if not pd.api.types.is_datetime64_any_dtype(df["time"]):
            error(f"[{symbol}] 'time' column is not datetime")
            return None

        # Create a copy to avoid modifying original
        df = df.copy()

        def _clean_price_series(s: pd.Series) -> pd.Series:
            s = s.replace(0, np.nan).astype(float)
            s = s.interpolate(method="linear", limit_direction="both")
            return s.ffill().bfill()

        for col in ["open", "high", "low", "close"]:
            df[col] = _clean_price_series(df[col])
            if df[col].isna().any():
                warn(f"[{symbol}] {col} still contains NaN after cleaning.")

        # Volatility threshold check
        vol_threshold = 1e-7 * df["close"].mean()
        close_diff_early = df["close"].head(2).diff().dropna()
        if len(close_diff_early) >= 1 and close_diff_early.abs().mean() < vol_threshold:
            warn(f"[{symbol}] Early close prices near-constant. Neutral signal.")
            df["signal"] = 0
            return df

        close_diff = df["close"].diff().dropna()
        if len(close_diff) >= norm_period and close_diff.abs().mean() < vol_threshold:
            warn(f"[{symbol}] Close prices near-constant. Neutral signal.")
            df["signal"] = 0
            return df

        # Calculate JMA (using EMA as approximation)
        price = df["close"].astype(float)
        alpha = 2.0 / (smooth_length + 1.0)
        jma = price.ewm(alpha=alpha, adjust=False).mean()
        df["jma"] = jma

        # Normalize JMA
        roll_mean = jma.rolling(window=norm_period, min_periods=1).mean().bfill().interpolate(method="linear")
        roll_std = jma.rolling(window=norm_period, min_periods=1).std(ddof=0).bfill().interpolate(method="linear").clip(lower=min_std)
        df["jma_roll_mean"] = roll_mean
        df["jma_roll_std"] = roll_std

        norm_jma = (jma - roll_mean) / roll_std
        norm_jma = norm_jma.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        df["norm_jma"] = norm_jma

        # Calculate RS volatility
        min_price = max(min_price, 1e-12)
        o = df["open"].clip(lower=min_price)
        h = df["high"].clip(lower=min_price)
        l = df["low"].clip(lower=min_price)
        c = df["close"].clip(lower=min_price)

        log_hc = np.log(h / c)
        log_ho = np.log(h / o)
        log_lc = np.log(l / c)
        log_lo = np.log(l / o)

        rs_combined = (log_hc * log_ho) + (log_lc * log_lo)
        rs_vol = rs_combined.rolling(window=rs_period, min_periods=1).mean() * 1e6
        df["rs_vol"] = rs_vol

        # Calculate volatility threshold
        min_vol_threshold = 1e-4
        rs_threshold = rs_vol.rolling(window=norm_period, min_periods=1).quantile(rs_percentile / 100.0)
        rs_threshold = rs_threshold.clip(lower=min_vol_threshold)
        df["rs_threshold"] = rs_threshold

        # Generate signals
        long_mask = (norm_jma > upper_threshold) & (rs_vol > rs_threshold)
        short_mask = (norm_jma < lower_threshold) & (rs_vol > rs_threshold)

        signals = pd.Series(0, index=df.index, dtype=np.int8)
        signals[long_mask] = 1
        signals[short_mask] = 2
        df["signal"] = signals

        # Log signal information
        if len(df) >= 3 and df.iloc[-2]["signal"] != 0:
            triggered = int(df.iloc[-2]["signal"])
            trigger_time = df.iloc[-2]["time"]
            debug(
                f"[{symbol}] Signal {triggered} ({'LONG' if triggered == 1 else 'SHORT'}) "
                f"triggered at {trigger_time}: "
                f"norm_jma={df['norm_jma'].iloc[-3]:.4f}, "
                f"rs_vol={df['rs_vol'].iloc[-3]:.4f}, "
                f"rs_threshold={df['rs_threshold'].iloc[-3]:.4f}"
            )

        return df

    except Exception as exc:
        error(f"[{symbol}] Indicator computation failed: {type(exc).__name__}: {str(exc)}", exc_info=True)
        return None

# ===================== Trading Logic =====================
class BinanceCryptoTrader:
    def __init__(self):
        self.client = BinanceClient()
        self.fee_rate = self.client.config.FEE_RATE  # Fixed: Access fee rate from config
        self.max_retries = 3

        # Extract base and quote assets from symbol
        self.base_asset = self.client.config.SYMBOL.replace('USDT', '').replace('BUSD', '').replace('BTC', '').replace('ETH', '')
        self.quote_asset = 'USDT'  # Assuming USDT pairs

        logger.info(f"Initialized BinanceCryptoTrader for {self.client.config.SYMBOL} ({'testnet' if self.client.config.TESTNET else 'live'} trading)")

    @property
    def config(self):
        return self.client.config

    def sync_position(self, symbol):
        """Sync position state with Binance balances"""
        base_free, base_locked = self.client.get_balance(self.base_asset)
        quote_free, quote_locked = self.client.get_balance(self.quote_asset)

        state = PositionState.load()
        open_orders = self.client.get_open_orders(symbol)

        if open_orders:
            logger.warning(f"[{symbol}] Found {len(open_orders)} open orders, canceling to sync state")
            self.client.cancel_all_orders(symbol)

        # Check if we have a position based on balance
        has_base_position = base_free > self.config.MIN_QTY

        if has_base_position and symbol not in state:
            logger.warning(f"[{symbol}] Base asset balance found but no local state. Manual position detected.")
            # For manual positions, we can't determine entry price, so we skip
            # Users should close manual positions before starting the bot
            logger.info(f"[{symbol}] Please close manual positions before starting automated trading")

        elif not has_base_position and symbol in state:
            logger.warning(f"[{symbol}] Local state exists but no base asset balance. Clearing state.")
            PositionState.clear(symbol)

    def estimate_slippage(self, symbol):
        bid, ask = self.client.get_orderbook(symbol)
        if bid > 0 and ask > 0:
            mid = (bid + ask) / 2.0
            slippage = (ask - bid) / mid
            logger.debug(f"[{symbol}] Estimated slippage: {slippage:.4%}")
            return slippage
        logger.warning(f"[{symbol}] Using conservative default slippage rate of 0.1%")
        return 0.001

    def calculate_quantity(self, symbol, entry_price):
        """Calculate position size based on available USDT balance"""
        try:
            quote_free, _ = self.client.get_balance(self.quote_asset)

            if quote_free <= 0:
                logger.error(f"[{symbol}] Insufficient {self.quote_asset} balance: {quote_free}")
                return 0.0

            # Use 10% of available balance with fee buffer
            fee_buffer = 1 - self.fee_rate * 2
            position_value = quote_free * 0.10 * fee_buffer
            qty = position_value / entry_price

            # Round to appropriate precision
            qty = self.client.round_quantity(qty)

            # Check minimum quantity and notional value
            if qty < self.config.MIN_QTY:
                logger.warning(f"[{symbol}] Calculated quantity {qty} below minimum {self.config.MIN_QTY}")
                return 0.0

            notional_value = qty * entry_price
            if notional_value < self.config.MIN_NOTIONAL:
                logger.warning(f"[{symbol}] Notional value {notional_value} below minimum {self.config.MIN_NOTIONAL}")
                return 0.0

            return qty

        except Exception as e:
            logger.error(f"[{symbol}] Quantity calculation failed: {type(e).__name__}: {str(e)}", exc_info=True)
            return 0.0

    def enter_position(self, symbol, direction, bar, slippage_rate):
        try:
            open_price = float(bar["open"])
            timestamp = bar["time"]

            if open_price < self.config.MIN_PRICE:
                logger.error(f"[{symbol}] Invalid bar price: open={open_price}")
                return

            self.client.cancel_all_orders(symbol)

            # For Binance, we can only go long (buy base asset)
            if direction == -1:
                logger.info(f"[{symbol}] Short signals not supported on Binance spot. Skipping.")
                return

            entry_price = open_price * (1 + slippage_rate)  # Add slippage for market buy
            qty = self.calculate_quantity(symbol, entry_price)

            if qty <= 0:
                logger.error(f"[{symbol}] Invalid quantity: {qty}")
                return

            for attempt in range(self.max_retries):
                try:
                    market_order = self.client.submit_market_order(symbol, qty, "buy")

                    if not self.client.confirm_order_filled(symbol, market_order['orderId']):
                        logger.warning(f"[{symbol}] Market order {market_order['orderId']} not filled, retrying...")
                        self.client.cancel_all_orders(symbol)
                        continue

                    # Get actual fill price from order
                    filled_order = self.client.client.get_order(symbol=symbol, orderId=market_order['orderId'])
                    actual_price = float(filled_order['price']) if filled_order['price'] != '0' else entry_price
                    actual_qty = float(filled_order['executedQty'])

                    logger.info(f"[{symbol}] Entered LONG at {actual_price:.4f}, qty={actual_qty}")

                    state = {
                        "entry_time": timestamp.isoformat(),
                        "entry_price": actual_price,
                        "quantity": actual_qty,
                        "direction": direction,
                        "stop_loss_price": actual_price * (1 - self.config.ENTRY_STOP_LOSS_PCT),
                        "best_high": actual_price,
                        "best_low": None,
                        "last_bar_time": timestamp.isoformat(),
                        "is_entry_candle": True,
                        "market_order_id": str(market_order['orderId'])
                    }
                    PositionState.update(symbol, state)
                    break

                except (InsufficientBalanceError, MinNotionalError) as e:
                    logger.error(f"[{symbol}] Entry failed: {e}")
                    return
                except Exception as e:
                    logger.error(f"[{symbol}] Entry attempt {attempt+1} failed: {type(e).__name__}: {str(e)}", exc_info=True)
                    self.client.cancel_all_orders(symbol)
                    if attempt == self.max_retries - 1:
                        logger.error(f"[{symbol}] Max retries reached for entry")
                        return

        except Exception as e:
            logger.error(f"[{symbol}] Enter position failed: {type(e).__name__}: {str(e)}", exc_info=True)

    def should_update_stop(self, symbol, bar, state):
        direction = int(state["direction"])
        high = float(bar["high"])
        low = float(bar["low"])

        if direction == 1:  # Long position
            best_high = float(state.get("best_high", high))
            return high > best_high
        else:  # Short position (not used in spot trading)
            best_low = float(state.get("best_low", low))
            return low < best_low

    def update_trailing(self, symbol, bar, state):
        direction = int(state["direction"])
        high = float(bar["high"])
        low = float(bar["low"])
        current_price = float(bar["close"])

        if direction == 1:  # Long position
            best_high = float(state.get("best_high", high))
            if high > best_high:
                best_high = high
                state["best_high"] = best_high
                new_stop_price = best_high * (1 - self.config.STOP_LOSS_PCT)

                if current_price <= new_stop_price:
                    self.exit_position(symbol, bar, self.estimate_slippage(symbol), reason="virtual stop hit")
                    return  # Exit early since position was closed
                else:
                    state["stop_loss_price"] = new_stop_price
                    logger.debug(f"[{symbol}] Updated trailing stop to {new_stop_price:.4f}")

        PositionState.update(symbol, state)

    def manage_position(self, symbol, closed_bar, new_bar, slippage_rate):
        state = PositionState.load().get(symbol)
        if not state:
            return False

        try:
            direction = int(state["direction"])
            timestamp = closed_bar["time"]

            # Skip management on entry candle
            if state.get("is_entry_candle", False):
                state["is_entry_candle"] = False
                state["last_bar_time"] = timestamp.isoformat()
                PositionState.update(symbol, state)
                return True

            # Check for stop loss hit first
            current_price = float(closed_bar["close"])
            stop_price = float(state.get("stop_loss_price", 0))

            if direction == 1 and current_price <= stop_price:
                self.exit_position(symbol, new_bar, slippage_rate, reason="stop loss hit")
                return False

            # Update trailing stop
            if self.should_update_stop(symbol, closed_bar, state):
                self.update_trailing(symbol, closed_bar, state)
                logger.info(f"[{symbol}] Updated trailing stop at time={timestamp}")
            else:
                logger.debug(f"[{symbol}] No favorable stop movement at time={timestamp}")

            # Update timestamp
            state = PositionState.load().get(symbol)  # Reload in case it was cleared
            if state:
                state["last_bar_time"] = timestamp.isoformat()
                PositionState.update(symbol, state)

            return True

        except Exception as e:
            logger.error(f"[{symbol}] Manage position failed: {type(e).__name__}: {str(e)}", exc_info=True)
            return True

    def exit_position(self, symbol, bar, slippage_rate, reason=""):
        state = PositionState.load().get(symbol)
        if not state:
            return

        # Check if we have base asset to sell
        base_free, _ = self.client.get_balance(self.base_asset)
        if base_free < self.config.MIN_QTY:
            logger.warning(f"[{symbol}] Insufficient {self.base_asset} balance for exit: {base_free}. Clearing state.")
            PositionState.clear(symbol)
            return

        try:
            direction = int(state["direction"])
            qty = min(float(state["quantity"]), base_free)  # Use minimum of state qty and available balance
            entry_price = float(state["entry_price"])
            open_price = float(bar["open"])
            timestamp = bar["time"]

            if open_price < self.config.MIN_PRICE:
                logger.error(f"[{symbol}] Invalid open price for exit: {open_price}")
                return

            self.client.cancel_all_orders(symbol)

            # Round quantity for exit
            qty = self.client.round_quantity(qty)
            exit_price = open_price * (1 - slippage_rate)  # Subtract slippage for market sell

            for attempt in range(self.max_retries):
                try:
                    order = self.client.submit_market_order(symbol, qty, "sell")

                    if self.client.confirm_order_filled(symbol, order['orderId']):
                        # Get actual fill details
                        filled_order = self.client.client.get_order(symbol=symbol, orderId=order['orderId'])
                        actual_exit_price = float(filled_order['price']) if filled_order['price'] != '0' else exit_price
                        actual_qty = float(filled_order['executedQty'])

                        pnl = (actual_exit_price - entry_price) * actual_qty * direction
                        fees = self.fee_rate * (entry_price + actual_exit_price) * actual_qty
                        net_pnl = pnl - fees

                        logger.info(f"[{symbol}] Exited at {actual_exit_price:.4f} due to {reason}, Net PnL={net_pnl:.2f} {self.quote_asset}")
                        PositionState.clear(symbol)
                        return
                    else:
                        logger.warning(f"[{symbol}] Exit order {order['orderId']} not filled, retrying...")
                        self.client.cancel_all_orders(symbol)

                except (InsufficientBalanceError, MinNotionalError) as e:
                    logger.error(f"[{symbol}] Exit failed: {e}")
                    return
                except Exception as e:
                    logger.error(f"[{symbol}] Exit attempt {attempt+1} failed: {type(e).__name__}: {str(e)}", exc_info=True)
                    self.client.cancel_all_orders(symbol)
                    if attempt == self.max_retries - 1:
                        logger.error(f"[{symbol}] Max exit retries reached.")
                        return

        except Exception as e:
            logger.error(f"[{symbol}] Exit position failed: {type(e).__name__}: {str(e)}", exc_info=True)

    def wait_for_next_bar(self):
        try:
            now = datetime.now(timezone.utc)
            minutes = (now.minute // 5 + 1) * 5
            next_bar = (now.replace(minute=0, second=0, microsecond=0) + timedelta(minutes=minutes)).replace(
                second=self.config.GRACE_SECONDS
            )
            sleep_seconds = max((next_bar - now).total_seconds(), 0)

            if sleep_seconds <= 0:
                sleep_seconds += 300

            logger.info(f"Waiting {sleep_seconds:.1f}s until {next_bar.isoformat()}")
            time.sleep(sleep_seconds)

        except Exception as e:
            logger.error(f"[{self.config.SYMBOL}] Wait for next bar failed: {type(e).__name__}: {str(e)}", exc_info=True)
            time.sleep(300)

    def get_fee_rate(self):
        return float(self.config.FEE_RATE)

    def run(self, max_cycles=float('inf')):
        logger.info(f"Starting BinanceCryptoTrader for {self.config.SYMBOL} ({'testnet' if self.config.TESTNET else 'live'} trading)")

        # Test connection first
        if not self.client.test_connection():
            logger.error("Connection test failed. Please fix API credentials before continuing.")
            return

        error_count = 0
        max_errors = 5
        cycle_count = 0
        last_rate_limit_time = None

        while cycle_count < max_cycles:
            try:
                # Rate limit handling
                if last_rate_limit_time and (datetime.now(timezone.utc) - last_rate_limit_time).total_seconds() < 60:
                    logger.warning(f"[{self.config.SYMBOL}] Rate limit detected recently, waiting 60s")
                    time.sleep(60)

                self.wait_for_next_bar()
                self.sync_position(self.config.SYMBOL)

                df = fetch_bars(self.client, self.config.SYMBOL, bars=self.config.BARS_LOOKBACK)
                if df is None or df.shape[0] < 60:
                    logger.warning(f"[{self.config.SYMBOL}] Insufficient or invalid data: {len(df) if df is not None else 0} bars")
                    error_count += 1
                    time.sleep(10)
                    continue

                slippage_rate = self.estimate_slippage(self.config.SYMBOL)
                state = PositionState.load().get(self.config.SYMBOL)
                position_open = bool(state)

                if position_open:
                    self.manage_position(self.config.SYMBOL, df.iloc[-2], df.iloc[-1], slippage_rate)
                else:
                    # Compute signals for new entry
                    df = compute_jma_signal(df, symbol=self.config.SYMBOL, logger=logger)
                    if df is None:
                        logger.warning(f"[{self.config.SYMBOL}] Indicator computation failed.")
                        error_count += 1
                        time.sleep(10)
                        continue

                    closed_bar = df.iloc[-2]
                    new_bar = df.iloc[-1]
                    signal = int(closed_bar["signal"])

                    logger.info(f"[{self.config.SYMBOL}] Checking signal: {signal} ({'LONG' if signal == 1 else 'SHORT' if signal == 2 else 'NEUTRAL'}) at {closed_bar['time']}")

                    # Only process long signals for spot trading
                    if signal == 1:
                        direction = 1
                        self.enter_position(
                            self.config.SYMBOL, direction, new_bar, slippage_rate
                        )
                    elif signal == 2:
                        logger.info(f"[{self.config.SYMBOL}] Short signal detected but not supported in spot trading")

                logger.info(f"Cycle {cycle_count+1} complete at {datetime.now(timezone.utc).isoformat()}")
                error_count = 0
                cycle_count += 1

            except KeyboardInterrupt:
                logger.info("Trading bot stopped by user.")
                self.client.cancel_all_orders(self.config.SYMBOL)
                break
            except Exception as e:
                error_count += 1
                error_msg = str(e).lower()

                if "rate limit" in error_msg or "too many requests" in error_msg:
                    last_rate_limit_time = datetime.now(timezone.utc)
                    logger.warning(f"[{self.config.SYMBOL}] Rate limit hit, waiting 60s")
                    time.sleep(60)
                    continue

                logger.error(f"Cycle error {error_count}/{max_errors}: type={type(e).__name__}, message={str(e)}", exc_info=True)

                if error_count >= max_errors:
                    logger.error("Max errors reached. Stopping bot.")
                    self.client.cancel_all_orders(self.config.SYMBOL)
                    break

                time.sleep(60)

        logger.info(f"Completed {cycle_count} cycles. Stopping bot.")
        self.client.cancel_all_orders(self.config.SYMBOL)

# ===================== Entry Point =====================
if __name__ == "__main__":
    try:
        # Install required packages first
        import subprocess
        import sys

        def install_package(package):
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

        try:
            from binance.client import Client
        except ImportError:
            print("Installing python-binance...")
            install_package("python-binance")
            from binance.client import Client

        trader = BinanceCryptoTrader()
        trader.run()

    except Exception as e:
        logger.error(f"Failed to start trader: {type(e).__name__}: {str(e)}", exc_info=True)
        raise